## 07 — Ensemble & Communicate
Kết hợp Prophet (60%) + SARIMAX (40%) và trực quan hóa kết quả cuối cùng.

In [ ]:
# Ensemble 60% Prophet + 40% SARIMAX
prophet_q2 = pred_prophet["yhat"].values
sarimax_q2 = pred_sarimax["yhat"].values

ensemble_yhat = 0.6 * prophet_q2 + 0.4 * sarimax_q2

ensemble = pd.DataFrame({
    "ds"     : months_q2,
    "tháng"  : months_label,
    "prophet": prophet_q2,
    "sarimax": sarimax_q2,
    "yhat"   : ensemble_yhat,
    "yhat_lower": pred_prophet["yhat_lower"].values,
    "yhat_upper": pred_prophet["yhat_upper"].values,
})

print("Ensemble Q2/2026:")
for _, r in ensemble.iterrows():
    print(f"  {r['tháng']}: {r['yhat']/1e9:.2f} tỷ  (Prophet {r['prophet']/1e9:.2f}T | SARIMAX {r['sarimax']/1e9:.2f}T)")

In [ ]:
# Biểu đồ đường — lịch sử + dự báo
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(training.ds, training.y / 1e9, "o-", linewidth=2, label="Lịch sử thực tế")
ax.plot(months_q2, prophet_q2  / 1e9, "s--", linewidth=1.5, markersize=8, label="Prophet")
ax.plot(months_q2, sarimax_q2  / 1e9, "^--", linewidth=1.5, markersize=8, label="SARIMAX")
ax.plot(months_q2, ensemble_yhat / 1e9, "D-", linewidth=2.5, markersize=10, label="Ensemble (chính thức)")
ax.fill_between(months_q2, ensemble.yhat_lower / 1e9, ensemble.yhat_upper / 1e9, alpha=0.15)
ax.axvline(pd.Timestamp("2026-04-01"), linestyle=":", alpha=0.5)
ax.set_title("Dự báo Doanh số Q2/2026 — Xe đạp Thống Nhất")
ax.set_xlabel("Tháng")
ax.set_ylabel("Doanh số (tỷ VND)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.4)
fig.tight_layout()
plt.show()

In [ ]:
# Lưu kết quả
import os
os.makedirs("output", exist_ok=True)
pred_prophet.to_csv("output/predictions_prophet.csv", index=False)
pred_sarimax.to_csv("output/predictions_sarimax.csv", index=False)
ensemble.to_csv("output/predictions_ensemble_c1.csv", index=False)
print("Đã lưu vào output/")

print()
print("=" * 50)
print("DỰ BÁO DOANH SỐ Q2/2026 — THỐNG NHẤT BIKE")
print("=" * 50)
for _, r in ensemble.iterrows():
    print(f"  {r['tháng']}: {r['yhat']/1e9:.2f} tỷ VND")
print(f"  Tổng Q2  : {ensemble['yhat'].sum()/1e9:.2f} tỷ VND")
print("=" * 50)